In [1]:
!pip install tensorflow opencv-python librosa moviepy scikit-learn tqdm torch

In [2]:
import os
import cv2
import numpy as np
import librosa
import tensorflow as tf
from tqdm import tqdm
from tensorflow.keras import layers, Model
from sklearn.model_selection import train_test_split

In [3]:
print("TensorFlow version:", tf.__version__)
print("Available GPUs:", tf.config.list_physical_devices('GPU'))

gpus = tf.config.list_physical_devices('GPU')

if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)

TensorFlow version: 2.19.0
Available GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [4]:
import kagglehub
path = kagglehub.dataset_download("shreyaty08/fakeavceleb")

100%|██████████| 6.18G/6.18G [01:04<00:00, 103MB/s] 

Extracting files...


In [5]:
BASE_PATH = "/kaggle/input/fakeavceleb/FakeAVCeleb_v1.2/FakeAVCeleb_v1.2"

In [6]:
FRAME_SIZE = 128
MAX_FRAMES = 10
AUDIO_LEN = 100

In [7]:
def extract_frames(video_path):

    cap = cv2.VideoCapture(video_path)

    frames = []

    while len(frames) < MAX_FRAMES:

        ret, frame = cap.read()

        if not ret:
            break

        frame = cv2.resize(frame,(FRAME_SIZE,FRAME_SIZE))
        frame = frame / 255.0

        frames.append(frame)

    cap.release()

    if len(frames) < MAX_FRAMES:
        return None

    return np.array(frames)

In [8]:
def extract_audio_features(video_path):

    try:
        y, sr = librosa.load(video_path, sr=16000)

        mfcc = librosa.feature.mfcc(
            y=y,
            sr=sr,
            n_mfcc=40
        )

        if mfcc.shape[1] < AUDIO_LEN:
            return None

        return mfcc[:,:AUDIO_LEN]

    except:
        return None

In [11]:
import os
found_base_path = None

search_locations = ['/kaggle/input/fakeavceleb', '/root/.cache/kagglehub']

for loc in search_locations:
    if not os.path.exists(loc):
        continue
    for root, dirs, files in os.walk(loc):
        if 'RealVideo-RealAudio' in dirs:
            found_base_path = os.path.join(root)
            print(f'Detected dataset base path: {found_base_path}')
            break
    if found_base_path: break

if not found_base_path:
    print('Could not find dataset folders. Please ensure cell 6Zhqdz8lydfb ran successfully.')

Detected dataset base path: /root/.cache/kagglehub/datasets/shreyaty08/fakeavceleb/versions/1/FakeAVCeleb_v1.2/FakeAVCeleb_v1.2


In [12]:
import os

BASE_PATH = "/root/.cache/kagglehub/datasets/shreyaty08/fakeavceleb/versions/1/FakeAVCeleb_v1.2/FakeAVCeleb_v1.2"

categories = {
    "RealVideo-RealAudio":0,
    "FakeVideo-RealAudio":1,
    "FakeVideo-FakeAudio":1,
    "RealVideo-FakeAudio":1
}

video_paths = []
labels = []

for folder,label in categories.items():

    folder_path = os.path.join(BASE_PATH,folder)

    if not os.path.exists(folder_path):
        continue

    for root,_,files in os.walk(folder_path):

        for f in files:

            if f.endswith((".mp4",".avi",".mov",".mkv")):

                video_paths.append(os.path.join(root,f))
                labels.append(label)

print("Total videos found:",len(video_paths))

Total videos found: 21560


In [13]:
train_paths, test_paths, train_labels, test_labels = train_test_split(
    video_paths,
    labels,
    test_size=0.2,
    random_state=42
)

In [14]:
def data_generator(paths,labels,batch_size=8):

    while True:

        for i in range(0,len(paths),batch_size):

            batch_paths = paths[i:i+batch_size]
            batch_labels = labels[i:i+batch_size]

            Xv = []
            Xa = []
            y = []

            for p,l in zip(batch_paths,batch_labels):

                frames = extract_frames(p)
                audio = extract_audio_features(p)

                if frames is None or audio is None:
                    continue

                Xv.append(frames)
                Xa.append(audio)
                y.append(l)

            if len(Xv) == 0:
                continue

            yield (np.array(Xv), np.array(Xa)), np.array(y)

In [15]:
video_input = layers.Input(
    shape=(MAX_FRAMES,FRAME_SIZE,FRAME_SIZE,3)
)

x = layers.TimeDistributed(
    layers.Conv2D(32,(3,3),activation="relu")
)(video_input)

x = layers.TimeDistributed(
    layers.MaxPooling2D()
)(x)

x = layers.TimeDistributed(
    layers.Flatten()
)(x)

x = layers.LSTM(64)(x)

video_branch = layers.Dense(64,activation="relu")(x)

In [16]:
audio_input = layers.Input(
    shape=(40,AUDIO_LEN)
)

y = layers.Conv1D(32,3,activation="relu")(audio_input)
y = layers.MaxPooling1D()(y)
y = layers.Flatten()(y)

audio_branch = layers.Dense(64,activation="relu")(y)

In [17]:
combined = layers.concatenate([video_branch,audio_branch])

z = layers.Dense(64,activation="relu")(combined)
output = layers.Dense(1,activation="sigmoid")(z)

model = Model(
    inputs=[video_input,audio_input],
    outputs=output
)

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 10, 128,   │          0 │ -                 │
│ (InputLayer)        │ 128, 3)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time_distributed    │ (None, 10, 126,   │        896 │ input_layer[0][0] │
│ (TimeDistributed)   │ 126, 32)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_1       │ (None, 40, 100)   │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time_distributed_1  │ (None, 10, 63,    │          0 │ time_distributed… │
│ (TimeDistributed)   │ 63, 32)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d (Conv1D)     │ (None, 38, 32)    │      9,632 │ input_layer_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time_distributed_2  │ (None, 10,        │          0 │ time_distributed… │
│ (TimeDistributed)   │ 127008)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d       │ (None, 19, 32)    │          0 │ conv1d[0][0]      │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ (None, 64)        │ 32,530,688 │ time_distributed… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_1 (Flatten) │ (None, 608)       │          0 │ max_pooling1d[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 64)        │      4,160 │ lstm[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 64)        │     38,976 │ flatten_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 128)       │          0 │ dense[0][0],      │
│ (Concatenate)       │                   │            │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 64)        │      8,256 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 1)         │         65 │ dense_2[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 32,592,673 (124.33 MB)

 Trainable params: 32,592,673 (124.33 MB)

 Non-trainable params: 0 (0.00 B)

In [18]:
import warnings
warnings.filterwarnings("ignore")

batch_size = 16

train_gen = data_generator(train_paths,train_labels,batch_size)
test_gen = data_generator(test_paths,test_labels,batch_size)

history = model.fit(
    train_gen,
    steps_per_epoch=len(train_paths)//batch_size,
    validation_data=test_gen,
    validation_steps=len(test_paths)//batch_size,
    epochs=5
)

Epoch 1/5
1078/1078 ━━━━━━━━━━━━━━━━━━━━ 3394s 3s/step - accuracy: 0.9716 - loss: 0.1927 - val_accuracy: 0.9764 - val_loss: 0.1073
Epoch 2/5
1078/1078 ━━━━━━━━━━━━━━━━━━━━ 3382s 3s/step - accuracy: 0.9750 - loss: 0.1167 - val_accuracy: 0.9764 - val_loss: 0.1038
Epoch 3/5
1078/1078 ━━━━━━━━━━━━━━━━━━━━ 3446s 3s/step - accuracy: 0.9751 - loss: 0.1203 - val_accuracy: 0.9766 - val_loss: 0.1034
Epoch 4/5
1078/1078 ━━━━━━━━━━━━━━━━━━━━ 3405s 3s/step - accuracy: 0.9755 - loss: 0.1023 - val_accuracy: 0.9766 - val_loss: 0.0953
Epoch 5/5
1078/1078 ━━━━━━━━━━━━━━━━━━━━ 3435s 3s/step - accuracy: 0.9754 - loss: 0.1014 - val_accuracy: 0.9766 - val_loss: 0.0950


In [19]:
model.save('video_deepfake_model.keras')
print('Model saved')

Model saved
